# 1. Base de datos de personas, hijos y nacimientos

Este cuaderno arma las dos bases que usa el análisis. La primera, `personas.parquet`, reúne a todas las personas de 15 a 55 años de las dieciséis Encuestas Nacionales de Hogares (ENAHO) de 2010 a 2025, con su empleo, sus ingresos y los hijos que viven con ellas. La segunda, `nacimientos.parquet`, tiene los nacimientos de las Estadísticas Vitales de 2009 a 2024, con la edad de la madre y si el bebé es su primer hijo, y sirve para comprobar que la encuesta identifica bien a las madres primerizas.

La ENAHO no pregunta quién es la madre o el padre de cada persona. Solo registra el parentesco de cada una con la jefatura del hogar, así que los hijos se asignan a sus padres con reglas de parentesco y de edad, como hace IPUMS con los censos (paso 4). Además, los nombres y los códigos de las variables cambian entre años, por lo que cada variable se busca por su etiqueta y sus códigos se revisan año por año.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat

# Microdatos del INEC, con las subcarpetas Enaho/ y Nacimientos/ (ver data/README.md)
DATOS_INEC = Path(os.environ.get("DATOS_INEC", "../data/raw/inec"))
SALIDA = Path("../data/processed")
ANIOS = range(2010, 2026)            # ENAHO 2010-2025
ANIOS_VITALES = range(2009, 2025)    # Estadísticas Vitales 2009-2024, el último año publicado

## 1. Ubicar el archivo de cada año

In [2]:
def archivos_enaho(carpeta: Path) -> dict[int, Path]:
    """Archivo .sav de cada año. El año sale del nombre del archivo, y la ENAHO 2013 puede venir
    en una subcarpeta."""
    enaho = carpeta / "Enaho"
    rutas = list(enaho.glob("*.sav")) + list((enaho / "2013_extract").glob("*.sav"))
    encontrados = {}
    for ruta in rutas:
        anio = re.search(r"(20\d\d)", ruta.name)
        if anio and int(anio.group(1)) in ANIOS:
            encontrados[int(anio.group(1))] = ruta
    return dict(sorted(encontrados.items()))


if not (DATOS_INEC / "Enaho").is_dir():
    raise FileNotFoundError(f"No está la carpeta {DATOS_INEC / 'Enaho'}. Copie ahí los microdatos "
                            "del INEC como explica data/README.md, o cambie DATOS_INEC en la "
                            "primera celda.")
archivos = archivos_enaho(DATOS_INEC)
faltan = sorted(set(ANIOS) - set(archivos))
assert not faltan, f"Falta la ENAHO de estos años: {faltan}"
pd.DataFrame({"archivo": [r.name for r in archivos.values()]},
             index=pd.Index(list(archivos), name="año"))

,archivo
año,
2010,ENAHO 2010.sav
2011,ENAHO 2011.sav
2012,ENAHO 2012.sav
2013,ENAHO 2013.sav
2014,ENAHO 2014.sav
2015,ENAHO 2015.sav
2016,ENAHO 2016.sav
2017,ENAHO 2017.sav
2018,ENAHO 2018.sav


## 2. Encontrar cada variable aunque cambie de nombre o de código

1. Algunas variables cambian de nombre. El estado conyugal se llama `a26` en 2010, `A25` en 2011 y 2012, y `A6` desde 2013, y en 2011 y 2012 la columna `A26` es otra pregunta. La condición de aseguramiento pasa de `A10` a `A11`. Por eso estas variables se buscan por su etiqueta.
2. Algunos códigos cambian de numeración. En 2010 «hijo o hijastro» es un solo código del parentesco con la jefatura y desde 2011 son dos, y lo mismo pasa con el aseguramiento. Por eso cada código se clasifica por el texto de su etiqueta.

`validar_codigos` revisa que los demás códigos usados signifiquen lo esperado en todos los años (por ejemplo, que `A4 = 2` sea «mujer»).

In [ ]:
def columna(columnas, nombre: str) -> str | None:
    """Busca una columna sin distinguir mayúsculas de minúsculas (A4 o a4)."""
    return {c.upper(): c for c in columnas}.get(nombre.upper())


def columna_por_etiqueta(meta, etiqueta: str) -> str:
    """Busca una columna por su etiqueta (el texto de la pregunta), no por su nombre."""
    etiquetas = meta.column_names_to_labels
    halladas = [c for c, e in etiquetas.items() if e and e.strip().lower() == etiqueta]
    if len(halladas) != 1:
        raise ValueError(f"La etiqueta «{etiqueta}» aparece en {len(halladas)} columnas: "
                         f"{halladas}")
    return halladas[0]


def grupo_parentesco(etiqueta: str) -> str:
    """Clasifica el parentesco con la jefatura del hogar según el texto de su etiqueta."""
    t = etiqueta.lower()
    reglas = [("niet", "nieto"), ("hij", "hijo"), ("yerno", "yerno o nuera"),
              ("nuera", "yerno o nuera"), ("jefe", "jefatura"), ("esposo", "cónyuge"),
              ("compañ", "cónyuge"), ("padre", "padre o madre"), ("madre", "padre o madre"),
              ("suegr", "suegro"), ("herman", "hermano"), ("cuñad", "cuñado"),
              ("no familiar", "otro no familiar"), ("otro familiar", "otro familiar"),
              ("servicio", "servicio doméstico"), ("pensionista", "pensionista"),
              ("ignorad", "ignorado")]
    for clave, grupo in reglas:
        if clave in t:
            return grupo
    raise ValueError(f"Parentesco sin clasificar: {etiqueta}")


def cotiza_por_su_trabajo(etiqueta: str) -> int:
    """1 si la persona está asegurada en la CCSS por su propio trabajo (asalariada, por convenio,
    por cuenta propia o voluntaria)."""
    t = etiqueta.lower()
    if t.startswith("familiar"):
        return 0
    return int(t.startswith("asalariad") or t.startswith("voluntari") or "convenio" in t
               or "cuenta propia" in t or "trabajador independiente" in t)


# Lo que debe decir la etiqueta de cada código que usa el análisis, en todos los años
CODIGOS_ESPERADOS = {
    "A4": {2: ("mujer",)},
    "CIVIL": {1: ("unión libre", "union libre"), 2: ("casad",), 3: ("divorciad",),
              4: ("separad",), 5: ("viud",), 6: ("solter",)},
    "CondAct": {1: ("ocupad",), 2: ("desemplead", "desocupad")},
    "Zona": {1: ("urban",)},
    "NivInst": {0: ("sin nivel",), 1: ("primaria incompleta",), 2: ("primaria completa",),
                3: ("secundaria",), 6: ("secundaria",), 7: ("superior",), 8: ("superior",),
                99: ("ignorad",)},
    "B4": {1: ("si", "sí")},
}


def validar_codigos(meta, columnas: dict[str, str], anio: int) -> None:
    errores = []
    for nombre, esperados in CODIGOS_ESPERADOS.items():
        etiquetas = {int(float(k)): str(v).lower()
                     for k, v in meta.variable_value_labels.get(columnas[nombre], {}).items()}
        for codigo, textos in esperados.items():
            if not any(t in etiquetas.get(codigo, "") for t in textos):
                errores.append(f"{nombre}={codigo} es «{etiquetas.get(codigo)}», se esperaba "
                               f"{textos}")
    if errores: 
        raise ValueError(f"ENAHO {anio}: " + "; ".join(errores))


def llave_hogar(df: pd.DataFrame) -> pd.Series:
    """Identificador del hogar, que une las columnas de vivienda y hogar que traiga cada año."""
    orden = ["ID_VIVIENDA", "nro_Vivienda", "SEGMENTO", "UPM", "CUESTIONARIO"]
    candidatas = [columna(df.columns, c) for c in orden if columna(df.columns, c)]
    utiles = [c for c in candidatas if df[c].nunique() > 2]
    hogar = columna(df.columns, "HOGAR")
    return df[utiles + ([hogar] if hogar else [])].astype("Int64").astype(str).agg("_".join, axis=1)

## 3. Leer cada año

Se leen todas las personas del hogar, incluidos los niños, porque hacen falta para saber quién tiene hijos. La tabla de control muestra, para cada año, la columna del estado conyugal que encontró la búsqueda por etiqueta y que cada hogar tiene exactamente una jefatura, condición necesaria para usar el parentesco.

In [4]:
def leer_anio(anio: int, ruta: Path) -> tuple[pd.DataFrame, dict]:
    meta = pyreadstat.read_sav(ruta, metadataonly=True)[1]
    cm = meta.column_names
    civil = columna_por_etiqueta(meta, "estado conyugal")
    aseguramiento = columna_por_etiqueta(meta, "condición de aseguramiento")
    region = columna_por_etiqueta(meta, "región de planificación")
    revisar = {"A4": columna(cm, "A4"), "CIVIL": civil, "CondAct": columna(cm, "CondAct"),
               "Zona": columna(cm, "Zona"), "NivInst": columna(cm, "NivInst"),
               "B4": columna(cm, "B4")}
    validar_codigos(meta, revisar, anio)

    pedidas = ["FACTOR", "ID_VIVIENDA", "nro_Vivienda", "SEGMENTO", "UPM", "CUESTIONARIO", "HOGAR",
               "A3", "A4", "A5", "CondAct", "B4", "NivInst", "Zona", "HorTotNorm", "ipnt", "isnt"]
    encontradas = [columna(cm, p) for p in pedidas] + [civil, aseguramiento, region]
    usar = list(dict.fromkeys(c for c in encontradas if c))
    df, meta = pyreadstat.read_sav(ruta, usecols=usar)

    def col(nombre):
        return columna(df.columns, nombre)

    parentesco = {k: grupo_parentesco(str(v))
                  for k, v in meta.variable_value_labels[col("A3")].items()}
    cotiza = {k: cotiza_por_su_trabajo(str(v))
              for k, v in meta.variable_value_labels[aseguramiento].items()}

    datos = pd.DataFrame({
        "anio": anio,
        "hogar": (str(anio) + "_" + llave_hogar(df)).values,
        "parentesco": df[col("A3")].map(parentesco).values,
        "mujer": (df[col("A4")] == 2).values,
        "edad": df[col("A5")].values,
        "factor": df[col("FACTOR")].values,
        "estado_conyugal": df[civil].values,
        "condicion_actividad": df[col("CondAct")].values,
        "ausente_temporal": (df[col("B4")] == 1).values,
        "nivel_instruccion": df[col("NivInst")].values,
        "zona": df[col("Zona")].values,
        "region": df[region].values,
        "horas": pd.to_numeric(df[col("HorTotNorm")], errors="coerce").values,
        "ingreso_principal": pd.to_numeric(df[col("ipnt")], errors="coerce").values,
        "ingreso_secundario": pd.to_numeric(df[col("isnt")], errors="coerce").values,
        "cotiza": df[aseguramiento].map(cotiza).values,
    })
    assert datos["parentesco"].notna().all(), f"ENAHO {anio}: parentesco sin clasificar"
    jefaturas = datos[datos["parentesco"] == "jefatura"].groupby("hogar").size()
    jefaturas = jefaturas.reindex(datos["hogar"].unique(), fill_value=0)
    control = {"año": anio, "personas": len(datos), "hogares": datos["hogar"].nunique(),
               "estado conyugal": civil, "aseguramiento": aseguramiento,
               "hogares con una sola jefatura (%)": round(100 * (jefaturas == 1).mean(), 2)}
    return datos, control


partes, controles = [], []
for anio, ruta in archivos.items():
    datos, control = leer_anio(anio, ruta)
    partes.append(datos)
    controles.append(control)
gente = pd.concat(partes, ignore_index=True)
gente["id"] = np.arange(len(gente))
controles = pd.DataFrame(controles).set_index("año")
assert (controles["hogares con una sola jefatura (%)"] == 100).all()
controles

,personas,hogares,estado conyugal,aseguramiento,hogares con una sola jefatura (%)
año,,,,,
2010,41184,11603,a26,a10,100.0
2011,40860,11721,A25,A10,100.0
2012,39390,11374,A25,A10,100.0
2013,38779,11219,A6,A11,100.0
2014,38399,11405,A6,A11,100.0
2015,37291,11277,A6,A11,100.0
2016,37006,11335,A6,A11,100.0
2017,34843,10712,A6,A11,100.0
2018,35096,10942,A6,A11,100.0


Todos los años pasan la revisión de códigos y cada hogar tiene exactamente una jefatura. La encuesta de 2020, levantada durante la pandemia, tiene menos hogares que las demás.

Así quedó clasificado el parentesco y la condición de aseguramiento en el primer y el último año.

In [5]:
for anio in (2010, 2025):
    meta = pyreadstat.read_sav(archivos[anio], metadataonly=True)[1]
    a3 = columna(meta.column_names, "A3")
    aseg = columna_por_etiqueta(meta, "condición de aseguramiento")
    print(f"ENAHO {anio}")
    print("  Parentesco:",
          {str(v): grupo_parentesco(str(v)) for v in meta.variable_value_labels[a3].values()})
    cotizan = [str(v) for v in meta.variable_value_labels[aseg].values()
               if cotiza_por_su_trabajo(str(v))]
    print("  Cotiza a la CCSS por su trabajo:", cotizan)

ENAHO 2010
  Parentesco: {'jefe (a)': 'jefatura', 'esposo(a) o compañero': 'cónyuge', 'hijo(a) o hijastro(a)': 'hijo', 'yerno, nuera': 'yerno o nuera', 'nieto(a)': 'nieto', 'padre, madre o suegro(a)': 'padre o madre', 'hermano(a)': 'hermano', 'cuñado(a)': 'cuñado', 'otro familiar': 'otro familiar', 'otro no familiar': 'otro no familiar', 'servicio doméstico o su familiar': 'servicio doméstico', 'pensionista': 'pensionista', 'ignorado': 'ignorado'}
  Cotiza a la CCSS por su trabajo: ['asalariado', 'mediante convenio', 'cuenta propia']
ENAHO 2025
  Parentesco: {'Jefe o jefa': 'jefatura', 'Esposo(a) o compañera(o)': 'cónyuge', 'Hijo(a)': 'hijo', 'Hijastro(a)': 'hijo', 'Yerno, nuera': 'yerno o nuera', 'Nieto(a)': 'nieto', 'Padre o madre': 'padre o madre', 'Suegro(a)': 'suegro', 'Hermano(a)': 'hermano', 'Cuñado(a)': 'cuñado', 'Otro familiar': 'otro familiar', 'Otro no familiar': 'otro no familiar', 'Servicio doméstico o su familiar': 'servicio doméstico', 'Pensionista o su familiar': 'pensi

## 4. Asignar cada hijo a su madre y a su padre

La encuesta dice cómo se relaciona cada persona con la jefatura del hogar, no con las demás. Con eso se deduce quiénes son padres de quién, con reglas parecidas a las que usa IPUMS para vincular a madres e hijos en los censos (Sobek y Kennedy, 2011).

| Parentesco con la jefatura | Quién puede ser su madre o su padre |
|---|---|
| hijo o hijastro | la jefatura o su cónyuge |
| nieto | un hijo, hija, yerno o nuera de la jefatura |
| hermano | el padre o la madre de la jefatura |
| la jefatura | su padre o su madre, si vive en el hogar |
| cónyuge o cuñado | un suegro o suegra de la jefatura |
| otro familiar | un hermano, cuñado, nieto u otro familiar de la jefatura |
| otro no familiar, servicio doméstico o pensionista | otra persona del mismo grupo |

Además, la madre debe tener entre 13 y 50 años más que el hijo, y el padre entre 13 y 65. Los hijos de la jefatura se asignan siempre a la jefatura y a su cónyuge. En los demás casos, el hijo se asigna solo si hay una única persona de ese sexo que puede ser su madre o su padre. Si hay dos o más (por ejemplo, dos hijas de la jefatura con edad para ser madres del mismo nieto), no se asigna, y esas personas quedan marcadas como *inciertas*. Como no se sabe si tienen hijos, más adelante no se usan como personas sin hijos.

In [6]:
REGLAS = {   # parentesco del hijo -> parentescos que pueden tener sus padres
    "hijo": ["jefatura", "cónyuge"],
    "nieto": ["hijo", "yerno o nuera"],
    "hermano": ["padre o madre"],
    "jefatura": ["padre o madre"],
    "cónyuge": ["suegro", "padre o madre"],
    "cuñado": ["suegro", "padre o madre"],
    "otro familiar": ["hermano", "cuñado", "otro familiar", "nieto"],
    "otro no familiar": ["otro no familiar"],
    "servicio doméstico": ["servicio doméstico"],
    "pensionista": ["pensionista"],
}

nombres_hijo = {"id": "id_hijo", "parentesco": "parentesco_hijo", "edad": "edad_hijo"}
hijos = gente[["id", "hogar", "parentesco", "edad"]].rename(columns=nombres_hijo)
candidatos = gente[["id", "hogar", "parentesco", "edad", "mujer"]]
pares = []
for parentesco_hijo, posibles in REGLAS.items():
    hijos_p = hijos[hijos["parentesco_hijo"] == parentesco_hijo]
    pares.append(hijos_p.merge(candidatos[candidatos["parentesco"].isin(posibles)], on="hogar"))
pares = pd.concat(pares, ignore_index=True)
pares["diferencia"] = pares["edad"] - pares["edad_hijo"]
maxima = np.where(pares["mujer"], 50, 65)
pares = pares[(pares["diferencia"] >= 13) & (pares["diferencia"] <= maxima)]

# Los hijos de la jefatura se asignan a la jefatura y a su cónyuge
directos = pares[pares["parentesco_hijo"] == "hijo"].assign(regla="hijo de la jefatura")
# En los demás parentescos, solo si hay un único candidato de ese sexo
otros = pares[pares["parentesco_hijo"] != "hijo"]
n_candidatos = otros.groupby(["id_hijo", "mujer"])["id"].transform("nunique")
unicos = otros[n_candidatos == 1].assign(regla="otro parentesco, candidato único")
vinculos = pd.concat([directos, unicos], ignore_index=True)
# Quienes podían ser madre o padre de un hijo que quedó sin asignar
inciertos = set(otros.loc[n_candidatos > 1, "id"])

resumen_vinculos = vinculos.groupby(["regla", "mujer"]).size().unstack()
resumen_vinculos = resumen_vinculos.rename(columns={True: "madre", False: "padre"})
resumen_vinculos

mujer,padre,madre
regla,,
hijo de la jefatura,156071,211995
"otro parentesco, candidato único",13581,26419


In [7]:
# Número de hijos de cada madre o padre y edades del mayor y del menor
por_progenitor = vinculos.groupby("id")["edad_hijo"].agg(n_hijos="size", edad_hijo_mayor="max",
                                                         edad_hijo_menor="min")
gente = gente.merge(por_progenitor, left_on="id", right_index=True, how="left")
gente["n_hijos"] = gente["n_hijos"].fillna(0).astype(int)
gente["incierto"] = gente["id"].isin(inciertos) & (gente["n_hijos"] == 0)


def cobertura_madres(d: pd.DataFrame) -> pd.Series:
    porcentaje = 100 * np.average(d["con_madre"], weights=d["factor"])
    return pd.Series({"niños": len(d), "con madre identificada (%)": porcentaje})


# ¿Qué parte de los niños de 0 a 10 años quedó con su madre identificada?
ninos = gente[gente["edad"] <= 10]
con_madre = ninos["id"].isin(vinculos.loc[vinculos["mujer"], "id_hijo"])
cobertura = (ninos.assign(con_madre=con_madre).groupby("parentesco")
             .apply(cobertura_madres, include_groups=False).sort_values("niños", ascending=False))
total = 100 * np.average(con_madre, weights=ninos["factor"])
print(f"Niños de 0 a 10 años con su madre identificada: {total:.1f} %")

# Se guarda para citarlo en el informe
TABLAS = Path("../outputs/tables")
TABLAS.mkdir(parents=True, exist_ok=True)
cobertura.assign(total=total).to_csv(TABLAS / "vinculos_madre_hijo.csv", encoding="utf-8-sig")
cobertura.round(1)

Niños de 0 a 10 años con su madre identificada: 89.5 %


,niños,con madre identificada (%)
parentesco,,
hijo,63681.0,98.6
nieto,16532.0,62.3
otro familiar,2016.0,62.2
otro no familiar,608.0,40.7
hermano,75.0,61.3
cuñado,18.0,45.6
servicio doméstico,12.0,87.5
pensionista,11.0,91.8
yerno o nuera,1.0,0.0


Casi todos los hijos de la jefatura quedan con su madre identificada. Los que no la tienen viven con su padre sin pareja o con otros familiares. Entre los nietos, una parte no tiene a su madre en el hogar (los crían los abuelos) y otra tiene dos posibles madres; ninguno de esos casos se asigna, para no adivinar.

## 5. Crear las variables del análisis

| Variable | Qué es |
|---|---|
| `empleo` | 1 si la persona trabajó la semana anterior o tiene un empleo del que estuvo ausente (vacaciones, incapacidad o licencia) |
| `trabajando` | 1 si trabajó la semana anterior, lo que excluye a quienes estaban ausentes, por ejemplo con licencia de maternidad |
| `participa` | 1 si está en la fuerza de trabajo, es decir, ocupada o buscando empleo |
| `horas` | Horas normales de trabajo por semana; 0 si no tiene empleo |
| `ingreso_relativo` | Ingreso laboral mensual (empleo principal y secundario) dividido entre el promedio de los ocupados de 20 a 55 años de ese año; 0 si no tiene empleo. Así los años se comparan sin ajustar por inflación |
| `empleo_formal`, `empleo_informal` | Ocupada que cotiza o no cotiza a la CCSS por su trabajo. Su suma es `empleo` |
| `n_hijos`, `edad_hijo_mayor` | Hijos que viven con la persona y edad del mayor |
| `edad_primer_hijo` | Su edad menos la de su hijo mayor en el hogar |
| `educacion`, `estado_civil`, `urbano`, `region` | Las características con las que se emparejan personas en el análisis |

In [8]:
def nivel_educativo(codigo: pd.Series) -> pd.Series:
    """Cuatro niveles, como en Kleven, Landais y Leite-Mariante (2025)."""
    condiciones = [codigo <= 1, codigo == 2, codigo.between(3, 6), codigo.between(7, 8)]
    niveles = ["Menos que primaria", "Primaria", "Secundaria", "Superior"]
    return pd.Series(np.select(condiciones, niveles, default=""), index=codigo.index)


def estado_civil(codigo: pd.Series) -> pd.Series:
    condiciones = [codigo.isin([1, 2]), codigo.isin([3, 4]), codigo == 5, codigo == 6]
    estados = ["Casada o en unión", "Separada o divorciada", "Viuda", "Soltera"]
    return pd.Series(np.select(condiciones, estados, default=""), index=codigo.index)


ocupada = gente["condicion_actividad"] == 1
gente["empleo"] = ocupada.astype(int)
gente["trabajando"] = (ocupada & ~gente["ausente_temporal"]).astype(int)
gente["participa"] = gente["condicion_actividad"].isin([1, 2]).astype(int)
gente["horas"] = np.where(ocupada, gente["horas"].fillna(0), 0.0)
ingreso_total = (gente["ingreso_principal"].fillna(0).clip(lower=0)
                 + gente["ingreso_secundario"].fillna(0).clip(lower=0))
gente["ingreso"] = np.where(ocupada, ingreso_total, 0.0)
referencia = gente[ocupada & gente["edad"].between(20, 55)].groupby("anio").apply(
    lambda d: np.average(d["ingreso"], weights=d["factor"]), include_groups=False)
gente["ingreso_relativo"] = gente["ingreso"] / gente["anio"].map(referencia)
gente["empleo_formal"] = (ocupada & (gente["cotiza"] == 1)).astype(int)
gente["empleo_informal"] = (ocupada & (gente["cotiza"] != 1)).astype(int)
gente["educacion"] = nivel_educativo(gente["nivel_instruccion"])
gente["estado_civil"] = estado_civil(gente["estado_conyugal"])
gente["urbano"] = (gente["zona"] == 1).astype(int)
gente["edad_primer_hijo"] = (gente["edad"] - gente["edad_hijo_mayor"]).where(gente["n_hijos"] > 0)

print(f"Horas faltantes entre las personas ocupadas: {gente.loc[ocupada, 'horas'].isna().sum()}")
print(f"Ocupados sin ingreso declarado: {100 * (gente.loc[ocupada, 'ingreso'] == 0).mean():.1f} %")
personas = (gente[gente["edad"].between(15, 55)]
            .drop(columns=["ingreso_principal", "ingreso_secundario"]).reset_index(drop=True))
print(f"Personas de 15 a 55 años: {len(personas):,}; sin dato de educación o estado civil: "
      f"{((personas['educacion'] == '') | (personas['estado_civil'] == '')).sum():,}")

Horas faltantes entre las personas ocupadas: 0
Ocupados sin ingreso declarado: 1.6 %
Personas de 15 a 55 años: 330,255; sin dato de educación o estado civil: 294


## 6. Nacimientos de las Estadísticas Vitales

Cada registro trae el año de nacimiento, la edad de la madre y cuántos hijos ha tenido contando al recién nacido (`Hijosten`). Un nacimiento es el primer hijo cuando `Hijosten = 1`. Se descartan los registros con la edad de la madre o el número de hijos ignorados.

In [9]:
nac, meta_nac = pyreadstat.read_sav(DATOS_INEC / "Nacimientos" / "Nacimientos2005-2024.sav",
                                    usecols=["Anonac", "Edadmad", "Hijosten"])
print({c: meta_nac.column_names_to_labels[c] for c in ["Anonac", "Edadmad", "Hijosten"]})
print("Códigos especiales:",
      {c: meta_nac.variable_value_labels.get(c) for c in ["Edadmad", "Hijosten"]})
validos = (nac["Anonac"].isin(ANIOS_VITALES) & nac["Edadmad"].between(10, 55)
           & nac["Hijosten"].between(1, 30))
nombres = {"Anonac": "anio", "Edadmad": "edad_madre", "Hijosten": "hijos_tenidos"}
nacimientos = nac[validos].rename(columns=nombres).astype(int).reset_index(drop=True)
assert sorted(nacimientos["anio"].unique()) == list(ANIOS_VITALES), "Faltan años"
primeros = int((nacimientos["hijos_tenidos"] == 1).sum())
descartados = int(nac["Anonac"].isin(ANIOS_VITALES).sum() - len(nacimientos))
print(f"Nacimientos {min(ANIOS_VITALES)}-{max(ANIOS_VITALES)}: {len(nacimientos):,}, de ellos "
      f"{primeros:,} primeros hijos; descartados por datos ignorados: {descartados:,}")

{'Anonac': 'Año de nacimiento', 'Edadmad': 'Edad de la madre', 'Hijosten': 'Hijos tenidos por la madre'}
Códigos especiales: {'Edadmad': {0.0: 'Ignorada'}, 'Hijosten': {99.0: 'Ignorado'}}
Nacimientos 2009-2024: 1,034,443, de ellos 429,693 primeros hijos; descartados por datos ignorados: 3,953


## 7. Resumen y guardado

In [10]:
def empleo_ponderado(d: pd.DataFrame) -> float:
    return 100 * np.average(d["empleo"], weights=d["factor"])


def resumen_anio(d: pd.DataFrame) -> pd.Series:
    mujeres, hombres = d[d["mujer"]], d[~d["mujer"]]
    return pd.Series({
        "personas": len(d),
        "madres": int((mujeres["n_hijos"] > 0).sum()),
        "padres": int((hombres["n_hijos"] > 0).sum()),
        "sin hijos": int(((d["n_hijos"] == 0) & ~d["incierto"]).sum()),
        "inciertas": int(d["incierto"].sum()),
        "empleo mujeres 20-45 (%)": empleo_ponderado(mujeres[mujeres["edad"].between(20, 45)]),
        "empleo hombres 20-45 (%)": empleo_ponderado(hombres[hombres["edad"].between(20, 45)]),
    })


resumen = personas.groupby("anio").apply(resumen_anio, include_groups=False)
enteros = ["personas", "madres", "padres", "sin hijos", "inciertas"]
resumen[enteros] = resumen[enteros].astype(int)
print(f"Total {min(ANIOS)}-{max(ANIOS)}: {len(personas):,} personas de 15 a 55 años; "
      f"{int(resumen['madres'].sum()):,} madres y {int(resumen['padres'].sum()):,} padres con "
      "hijos en el hogar")
resumen.round(1)

Total 2010-2025: 330,255 personas de 15 a 55 años; 98,437 madres y 65,506 padres con hijos en el hogar


,personas,madres,padres,sin hijos,inciertas,empleo mujeres 20-45 (%),empleo hombres 20-45 (%)
anio,,,,,,,
2010,25111,7772,5597,10780,962,53.7,85.7
2011,24994,7650,5484,10926,934,55.5,86.5
2012,24162,7365,5203,10602,992,55.7,84.9
2013,23575,7178,5035,10406,956,55.1,84.0
2014,23484,6984,4817,10802,881,54.6,83.9
2015,22616,6845,4687,10153,931,55.0,84.1
2016,22228,6737,4547,10139,805,56.4,84.5
2017,20647,6279,4169,9503,696,54.7,84.1
2018,20616,6241,4008,9710,657,54.9,82.4


In [11]:
SALIDA.mkdir(parents=True, exist_ok=True)
personas.to_parquet(SALIDA / "personas.parquet", index=False)
nacimientos.to_parquet(SALIDA / "nacimientos.parquet", index=False)
print(f"Guardado: {len(personas):,} personas con {personas.shape[1]} variables y "
      f"{len(nacimientos):,} nacimientos.")

Guardado: 330,255 personas con 30 variables y 1,034,443 nacimientos.
